# AM-KG walkthrough

This notebook runs the whole pipeline on one paper, step by step: parse → extract → normalise → quality control → graph → query → assistant.

> The demo papers are **DEMO / SYNTHETIC** and are not real research. Replace `PAPER` with a real PDF path for real work.

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
from src.config import get_settings
S = get_settings()
PAPER = S.path("papers_dir") / "demo_01_lpbf_ti64.pdf"
if not PAPER.exists():
    from src.ingestion.demo_pdf import build_demo_pdfs
    build_demo_pdfs(S.path("demo_dir"), S.path("papers_dir"))
PAPER

## 1. Parse the PDF: metadata, sections, page-tagged sentences, tables

In [ ]:
from src.ingestion.loader import parse_document
doc = parse_document(PAPER, S)
print(doc.paper.title, doc.paper.year, doc.paper.doi, "synthetic:", doc.paper.is_synthetic)
print([(s.section_type, s.title) for s in doc.sections])
print(doc.tables[0].header if doc.tables else "no tables")
for s in doc.sentences[:3]: print(s.page, s.section_type, "|", s.text)

## 2. Rule-based entities and values for one sentence

In [ ]:
from src.normalization.lexicon import Lexicon
from src.normalization.entity_normalizer import EntityNormalizer
from src.entity_extraction.rule_based import RuleEntityExtractor
lex = Lexicon.load(S.path("lexicon_file")); norm = EntityNormalizer(lex)
ex = RuleEntityExtractor(lex, norm)
sent = [s for s in doc.sentences if "manufactured using" in s.text][0]
an = ex.analyze(sent, doc.paper.paper_id)
for m in an.mentions:
    print(f"{m.text!r:28} {m.entity_type:16} -> {m.canonical_name!r:22} norm={m.normalization_confidence}",
          f"value={m.quantity.raw!r} -> {m.quantity.normalized_value} {m.quantity.normalized_unit}" if m.quantity else "")

## 3. Full extraction (rule mode here; use `mode='hybrid'` when Ollama is running)

In [ ]:
from src.pipeline import ExtractionPipeline
pipe = ExtractionPipeline(S, mode="rule")
res = pipe.process_file(PAPER)
print(res.stats)
import pandas as pd
pd.DataFrame([{"status": r.status.value, "conf": r.confidence, "type": r.knowledge_type.value,
               "subject": r.subject.name, "relation": r.relation, "object": r.object.name,
               "page": r.evidence.page, "section": r.evidence.section, "evidence": r.evidence.text[:90]}
              for r in res.relations]).head(25)

## 4. Normalisation examples (aliases, units)

In [ ]:
for s, t in [("SLM", "Process"), ("Selective laser melting", "Process"), ("Ti–6Al–4V", "Material"),
             ("DMLS", "Process"), ("SLS", "Process"), ("preheat temp", "Parameter")]:
    r = norm.normalize(s, t); print(f"{s!r:28} -> {r.canonical_name!r:24} conf={r.confidence} ({r.method})")
from src.normalization.units import parse_quantity_text
for v in ["0.2 kW", "30 µm", "0.03 mm", "1.2 m/s", "1073 K"]:
    q = parse_quantity_text(v); print(v, "->", q.normalized_value, q.normalized_unit)

## 5. Build the knowledge graph for all extracted papers and run queries

In [ ]:
pipe.process_many(S.path("papers_dir"))
from src.knowledge_graph.kb import KnowledgeBase
from src.knowledge_graph.store import get_store
kb = KnowledgeBase.load(S)
store = get_store(S, kb)      # Neo4j if NEO4J_URI is set, else local store
print(store.name, store.build())
pd.DataFrame(store.run_query("studies_linking", a="laser power", b="porosity"))[["paper_id","relation","trend","page","section","evidence"]]

## 6. Why does the KG believe this claim?

In [ ]:
cid = store.run_query("studies_linking", a="laser power", b="porosity")[0]["claim_id"]
store.run_query("why_claim", claim_id=cid)

## 7. Research assistant (graph-only here; uses Qwen automatically when Ollama is running)

In [ ]:
from src.retrieval.assistant import ResearchAssistant
ans = ResearchAssistant(kb, S, use_llm=False).answer("What parameters affect porosity in LPBF Ti-6Al-4V?")
print(ans.answer); print(ans.warnings)

## 8. Experimental: potentially under-studied combinations

In [ ]:
from src.analysis.gaps import find_gaps
rep = find_gaps(kb)
for c in rep.candidates[:5]: print("-", c.statement)
rep.caveats